<a href="https://colab.research.google.com/github/sandesh-py/ML2/blob/main/Program_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import math

# --------------------------------------------------
# PLAY TENNIS DATASET
# --------------------------------------------------

data = {
    'Outlook': [
        'Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain',
        'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain',
        'Sunny', 'Overcast', 'Overcast', 'Rain'
    ],

    'Temperature': [
        'Hot', 'Hot', 'Hot', 'Mild', 'Cool',
        'Cool', 'Cool', 'Mild', 'Cool', 'Mild',
        'Mild', 'Mild', 'Hot', 'Mild'
    ],

    'Humidity': [
        'High', 'High', 'High', 'High', 'Normal',
        'Normal', 'Normal', 'High', 'Normal', 'Normal',
        'Normal', 'High', 'Normal', 'High'
    ],

    'Wind': [
        'Weak', 'Strong', 'Weak', 'Weak', 'Weak',
        'Strong', 'Strong', 'Weak', 'Weak', 'Weak',
        'Strong', 'Strong', 'Weak', 'Strong'
    ],

    'Play': [
        'No', 'No', 'Yes', 'Yes', 'Yes',
        'No', 'Yes', 'No', 'Yes', 'Yes',
        'Yes', 'Yes', 'Yes', 'No'
    ]
}

df = pd.DataFrame(data)


# --------------------------------------------------
# FOIL INFORMATION GAIN
# --------------------------------------------------

def foil_gain(p0, n0, p1, n1):

    if p1 == 0:
        return -999

    def log2(x):
        if x <= 0:
            return 0
        return math.log2(x)

    before = p0 * (
        log2(p0 / (p0 + n0))
    )

    after = p1 * (
        log2(p1 / (p1 + n1))
    )

    return after - before


# --------------------------------------------------
# FIND BEST CONDITION
# --------------------------------------------------

def find_best_condition(df):

    positive = df[df['Play'] == 'Yes']
    negative = df[df['Play'] == 'No']

    p0 = len(positive)
    n0 = len(negative)

    best_condition = None
    best_gain = -999

    for column in df.columns:

        if column == 'Play':
            continue

        for value in df[column].unique():

            pos_subset = positive[
                positive[column] == value
            ]

            neg_subset = negative[
                negative[column] == value
            ]

            p1 = len(pos_subset)
            n1 = len(neg_subset)

            gain = foil_gain(
                p0, n0,
                p1, n1
            )

            if gain > best_gain:
                best_gain = gain
                best_condition = (column, value)

    return best_condition, best_gain


# --------------------------------------------------
# FOIL
# --------------------------------------------------

def foil():

    remaining = df.copy()

    rules = []

    while len(
        remaining[remaining['Play'] == 'Yes']
    ) > 0:

        condition, gain = find_best_condition(
            remaining
        )

        column, value = condition

        rule_data = remaining[
            remaining[column] == value
        ]

        positive_count = len(
            rule_data[rule_data['Play'] == 'Yes']
        )

        negative_count = len(
            rule_data[rule_data['Play'] == 'No']
        )

        if positive_count > negative_count:

            rule = f"IF {column} = {value} THEN Play = Yes"

            rules.append(rule)

            remaining = remaining[
                ~(
                    (remaining[column] == value) &
                    (remaining['Play'] == 'Yes')
                )
            ]

        else:
            break

    return rules


# --------------------------------------------------
# DISPLAY FOIL RULES
# --------------------------------------------------

print("\n================ FOIL RULES ================\n")

rules = foil()

for i, rule in enumerate(rules, 1):
    print(f"Rule {i}: {rule}")


================ FOIL RULES ================

Rule 1: IF Outlook = Overcast THEN Play = Yes
Rule 2: IF Temperature = Cool THEN Play = Yes
Rule 3: IF Humidity = Normal THEN Play = Yes
